In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict  
from langchain.chat_models import init_chat_model # langchain 은 langgraph의 자매 프로젝트 같은 것, 단지 ai 모델이랑 쉽게 대화할 수 잇게 해주는
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages # 이것도 operator.add처럼 이전 메시지랑 새 메시지를 합쳐주는 거지만 추가로 memory 안의 메시지를 수정, 삭제할 수도 잇게 해준다.
from typing import Annotated
from langgraph.graph.message import MessagesState

llm = init_chat_model("openai:gpt-4o-mini") # 선택사항 이라함.. 

# llm.invoke([{"role":"user", "content":"Hello!"}]) # 깔끔하게 정돈된 형태로 ?나온다.

In [ ]:
class State(TypedDict): # state에는 메모리 같은 걸 만듦 
    # 3. reducer(add_messages)함수가 이전 메시지와 새 메시지를 합쳐줌 
    messages : Annotated[list[AnyMessage], add_messages ] # operator.add랑 유사하지만 메모리의 대화수정 삭제 기능 가능 

# class State(MessagesState):
#     pass  # 이렇게 하나 위처럼 하나 같다. 

graph_builder = StateGraph[State, None, State, State](State)

In [9]:
def chatbot(state : State):
    response = llm.invoke(state["messages"]) # 1. 응답은 메시지 하나로 
    return {
        "messages" : [response] # 2. 메시지 키를 가진 객체 반환 
    }

In [10]:
graph_builder.add_node("chatbot", chatbot)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

In [11]:
graph = graph_builder.compile()

graph.invoke(
    {
        "messages" : [
            {"role":"user","content" : "how are you?"}
        ]
    }
)

{'messages': [HumanMessage(content='how are you?', additional_kwargs={}, response_metadata={}, id='ee180d23-5eea-4ea9-9b58-b0cde01b4473'),
  AIMessage(content="I'm just a computer program, so I don't have feelings, but I'm here and ready to help you! How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 11, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'chatcmpl-D9NGRrZ1LKNSjnEpKjPKMtXlJgl7K', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c5f61-7ed6-7103-8083-61e82f02c94b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 28, '